# GUS04G — Visualization of Estimated Cross Tables

**Purpose:** Load the enriched database (`geoteryt_E.pkl`) produced by
GUS04F, randomly draw one administrative unit at each level (gmina,
powiat, old voivodeship, new voivodeship), and produce time-series
plots for every E\_ cross-table subject on each drawn unit.

**Observed vs. Estimated markers:**
- *Observed* (●, filled) — the source M\_ anchor subject has real
  (non-NaN) census data for that territory × year.
- *Estimated* (○, hollow) — value was produced by the estimation
  pipeline (interpolation / IPF / Gurobi QP).
- For powiats and voivodeships all cross-table values come from
  aggregation, so they are always shown as *Estimated*.

**Old voivodeship aggregation:** Pre-1999 voivodeships are not stored
as TERYTRecords in the hierarchy.  We aggregate E\_ tables from all
gminas that share the same `old_woj` attribute.

---

In [1]:
# ── Cell 1: Imports & database load ────────────────────────────────────
import os, sys, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
TOOLS_PATH = os.path.join(REPO, 'Code', 'tools')
if TOOLS_PATH not in sys.path:
    sys.path.insert(0, TOOLS_PATH)

from geoTERYT_db import (
    load_complete_database,
    LEVEL_GMINA, LEVEL_POWIAT, LEVEL_VOIVODESHIP,
    RODZ_AGGREGATION_SET,
)

DATA_ROOT = os.path.join(REPO, '..', '..', 'Data', 'Geospatial')
DB_PATH = os.path.join(DATA_ROOT, 'geoteryt_E.pkl')
assert os.path.isfile(DB_PATH), f'Database not found: {DB_PATH}'

db = load_complete_database(DB_PATH)
print(f'Loaded database with {len(db._records)} records.')

Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper/../../Data/Geospatial/geoteryt_E.pkl...
  Database version: 4.3
  ✓ Restored old voivodships: 49 rows
  ✓ Restored geometry store: 13,287 unique geometries


KeyboardInterrupt: 

In [ ]:
# ── Cell 2: Configuration — E_ subjects and their M_ anchor mapping ───
#
# E_SUBJECT_ID  →  list of source M_ subject IDs that contain
#                   *observed* census data at gmina level.
#
# If ANY anchor subject has non-NaN data at (teryt_id, year),
# we mark that point as "observed".

from pathlib import Path

E_TO_ANCHOR = {
    'E_age_sex_2000':  ['M_age_sex'],
    'E_age_sex_1990':  ['M_age_sex', 'M_age_1990'],
    'E_educ_2000':     ['M_educ_2000'],
    'E_educ_1990':     ['M_educ_1990'],
    'E_educ_sex_2000': ['M_educ_sex_2000'],
    'E_educ_sex_1990': ['M_educ_sex_1990'],
    'E_hh_size_2000':  ['M_hh_size_2000'],
    'E_hh_size_1990':  ['M_hh_size_1990'],
}

# Year ranges for each section
SECTION_YEARS = {
    '2000': list(range(1999, 2026)),  # 1999–2025
    '1990': list(range(1986, 2003)),  # 1986–2002
}

def get_section(e_sid: str) -> str:
    """Return '2000' or '1990' from the E_ subject ID."""
    return '2000' if '2000' in e_sid else '1990'

# Readable labels (for plot titles)
E_LABELS = {
    'E_age_sex_2000':  'Age × Sex (1999–2025)',
    'E_age_sex_1990':  'Age × Sex (1986–2002)',
    'E_educ_2000':     'Education (1999–2025)',
    'E_educ_1990':     'Education (1986–2002)',
    'E_educ_sex_2000': 'Education × Sex (1999–2025)',
    'E_educ_sex_1990': 'Education × Sex (1986–2002)',
    'E_hh_size_2000':  'Household Size (1999–2025)',
    'E_hh_size_1990':  'Household Size (1986–2002)',
}

# Random seed for reproducibility within a session
SEED = 6547
random.seed(SEED)
np.random.seed(SEED)

print('Configuration ready.  E_ subjects:', list(E_TO_ANCHOR.keys()))

plot_root = Path(REPO).parent.parent / 'Plots' / 'DB_visualization'
plot_root.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Cell 3: Helper — determine observed/estimated status ──────────────

def is_year_observed(record, e_sid: str, year: int) -> bool:
    """Return True if this year is marked as observed in the E_ CrossTable.

    Uses the ``observed_years`` attribute persisted on CrossTable objects
    by the estimation pipeline (Fix 39).  Falls back to probing the raw
    M_ anchor tables when ``observed_years`` is empty (backward compat).
    Works at ALL territorial levels (gmina, powiat, voivodeship, old-woj).
    """
    # ── primary: check observed_years on the E_ cross table itself ──
    e_ct = record.cross_tables.get(e_sid)
    if e_ct is not None and hasattr(e_ct, 'observed_years') and e_ct.observed_years:
        return year in e_ct.observed_years

    # ── fallback: probe M_ anchor tables (pre-Fix-39 databases) ──
    anchors = E_TO_ANCHOR.get(e_sid, [])
    for m_sid in anchors:
        ct = record.cross_tables.get(m_sid)
        if ct is None:
            continue
        tbl = ct.tables.get(year)
        if tbl is not None and not np.all(np.isnan(tbl)):
            return True
    return False


def _get_core_slices(ct):
    """Return non-ogółem indices for each dimension of a CrossTable.

    For dimensions that contain an 'ogółem' label, the returned list
    excludes that index.  For dimensions without it, all indices are kept.
    """
    core = []
    for di, dn in enumerate(ct.dim_names):
        labels = ct.dim_labels[dn]
        non_og = [i for i, lbl in enumerate(labels)
                  if 'ogółem' not in lbl.lower()]
        core.append(non_og if len(non_og) < len(labels) else list(range(len(labels))))
    return core


def _core_sum(tbl, core_slices, ndim):
    """Sum only the core (non-ogółem) cells of a table."""
    if ndim == 1:
        return float(np.nansum(tbl[core_slices[0]]))
    elif ndim == 2:
        return float(np.nansum(tbl[np.ix_(core_slices[0], core_slices[1])]))
    else:
        return float(np.nansum(tbl))  # 3-D fallback


def build_time_series(record, e_sid: str):
    """Extract the per-year time series from an E_ CrossTable.

    Returns
    -------
    years : list[int]
    totals : np.ndarray, shape (n_years,)
        Sum of **core** cells only (ogółem marginals excluded).
    observed_mask : np.ndarray[bool], shape (n_years,)
        True where the *corresponding anchor* has real data.
    """
    ct = record.cross_tables.get(e_sid)
    if ct is None:
        return [], np.array([]), np.array([], dtype=bool)
    years = sorted(ct.years_with_data)
    if not years:
        return [], np.array([]), np.array([], dtype=bool)
    core_sl = _get_core_slices(ct)
    totals = np.array([_core_sum(ct.tables[y], core_sl, ct.ndim) for y in years])
    obs = np.array([is_year_observed(record, e_sid, y) for y in years])
    return years, totals, obs


def build_category_series(record, e_sid: str, dim_idx: int = 0):
    """Extract per-category time series along `dim_idx`.

    For a 2-D cross table (e.g. age × sex), summing over *all other*
    dimensions uses only **non-ogółem** indices along each sum axis,
    so that marginals don't inflate the values.

    Returns
    -------
    years : list[int]
    labels : list[str]  – category labels along the chosen dimension
    values : np.ndarray, shape (n_labels, n_years)
    observed_mask : np.ndarray[bool], shape (n_years,)
    """
    ct = record.cross_tables.get(e_sid)
    if ct is None:
        return [], [], np.empty((0, 0)), np.array([], dtype=bool)
    years = sorted(ct.years_with_data)
    if not years:
        return [], [], np.empty((0, 0)), np.array([], dtype=bool)

    dim_name = ct.dim_names[dim_idx]
    labels = ct.dim_labels[dim_name]
    n_labels = len(labels)

    core_sl = _get_core_slices(ct)

    values = np.full((n_labels, len(years)), np.nan)
    for j, y in enumerate(years):
        tbl = ct.tables[y]
        if ct.ndim == 1:
            # 1-D: no sum axes, just use the values directly
            values[:, j] = tbl
        elif ct.ndim == 2:
            # 2-D: sum over the other dimension using only non-ogółem idx
            other_dim = 1 - dim_idx
            core_other = core_sl[other_dim]
            if dim_idx == 0:
                # marginal[i] = sum of tbl[i, core_cols]
                values[:, j] = np.nansum(tbl[:, core_other], axis=1)
            else:
                # marginal[j] = sum of tbl[core_rows, j]
                values[:, j] = np.nansum(tbl[core_other, :], axis=0)
        else:
            # 3-D+ fallback: original behaviour
            sum_axes = tuple(i for i in range(ct.ndim) if i != dim_idx)
            values[:, j] = np.nansum(tbl, axis=sum_axes)

    obs = np.array([is_year_observed(record, e_sid, y) for y in years])
    return years, labels, values, obs

print('Helpers defined.')

In [ ]:
# ── Cell 4: Random territory sampling ─────────────────────────────────
#
# 1. gmina   – level=6, rodz ∈ {1,2,3}, has at least one E_ subject
# 2. powiat  – level=5, has at least one E_ subject
# 3. new voivodeship – level=2, has at least one E_ subject
# 4. old voivodeship – unique old_woj name; we aggregate on the fly

e_sids = list(E_TO_ANCHOR.keys())

def has_any_e_subject(record):
    return any(sid in record.cross_tables for sid in e_sids)

# ── Gmina ──
gmina_candidates = [
    r for r in db._records.values()
    if r.level == LEVEL_GMINA
    and r.rodz in RODZ_AGGREGATION_SET
    and has_any_e_subject(r)
]
assert gmina_candidates, 'No eligible gminas found with E_ subjects!'
sampled_gmina = random.choice(gmina_candidates)

# ── Powiat ──
powiat_candidates = [
    r for r in db._records.values()
    if r.level == LEVEL_POWIAT
    and has_any_e_subject(r)
]
assert powiat_candidates, 'No eligible powiats found with E_ subjects!'
sampled_powiat = random.choice(powiat_candidates)

# ── New voivodeship ──
voiv_candidates = [
    r for r in db._records.values()
    if r.level == LEVEL_VOIVODESHIP
    and has_any_e_subject(r)
]
assert voiv_candidates, 'No eligible voivodeships found with E_ subjects!'
sampled_voiv = random.choice(voiv_candidates)

# ── Old voivodeship ──
# Collect distinct old_woj names from gminas with E_ data
old_woj_names = sorted({
    r.old_woj for r in db._records.values()
    if r.level == LEVEL_GMINA
    and r.rodz in RODZ_AGGREGATION_SET
    and r.old_woj is not None
    and has_any_e_subject(r)
})
assert old_woj_names, 'No old voivodeships found!'
sampled_old_woj = random.choice(old_woj_names)

print(f'Sampled gmina:            {sampled_gmina.name} ({sampled_gmina.teryt_id})')
print(f'Sampled powiat:           {sampled_powiat.name} ({sampled_powiat.teryt_id})')
print(f'Sampled new voivodeship:  {sampled_voiv.name} ({sampled_voiv.teryt_id})')
print(f'Sampled old voivodeship:  {sampled_old_woj}')

In [ ]:
# ── Cell 5: Aggregate E_ tables for the old voivodeship ───────────────
#
# Build a synthetic "record-like" object that holds aggregated
# cross tables for all gminas sharing `old_woj == sampled_old_woj`.

from types import SimpleNamespace
from geoTERYT_db import CrossTable

def aggregate_old_voivodeship(db, old_woj_name: str, e_sids: list):
    """Sum E_ cross tables across all gminas with a given old_woj.

    Returns a SimpleNamespace with:
      .name          – voivodeship name
      .teryt_id      – 'old_<name>'
      .cross_tables  – {e_sid: CrossTable}
    """
    gminas = [
        r for r in db._records.values()
        if r.level == LEVEL_GMINA
        and r.rodz in RODZ_AGGREGATION_SET
        and r.old_woj == old_woj_name
    ]
    agg = SimpleNamespace(
        name=old_woj_name,
        teryt_id=f'old_{old_woj_name}',
        level=None,  # synthetic
        cross_tables={},
    )
    for e_sid in e_sids:
        running_total = None
        # Collect observed_years from gminas (union = observed at old-woj level)
        all_observed_years: set = set()
        for g in gminas:
            ct = g.cross_tables.get(e_sid)
            if ct is None:
                continue
            # Gather observed years from gmina E_ cross table
            if hasattr(ct, 'observed_years'):
                all_observed_years |= ct.observed_years
            if running_total is None:
                running_total = CrossTable(
                    subject_id=ct.subject_id,
                    dim_names=ct.dim_names,
                    dim_labels=ct.dim_labels,
                    subject_name=ct.subject_name,
                    year_range=ct.year_range,
                )
                # Copy tables from first gmina
                for year in ct.year_range:
                    tbl = ct.tables.get(year)
                    if tbl is not None:
                        running_total.tables[year] = tbl.copy()
            else:
                # Add element-wise (NaN-safe: NaN + NaN → NaN,
                #                             NaN + x   → x)
                for year in ct.year_range:
                    a = running_total.tables.get(year)
                    b = ct.tables.get(year)
                    if a is None or b is None:
                        continue
                    both_nan = np.isnan(a) & np.isnan(b)
                    summed = np.where(np.isnan(a), 0, a) + np.where(np.isnan(b), 0, b)
                    summed[both_nan] = np.nan
                    running_total.tables[year] = summed
        if running_total is not None:
            # Set observed_years on the aggregated cross table
            running_total.observed_years = all_observed_years
            agg.cross_tables[e_sid] = running_total

    print(f'  Aggregated {len(gminas)} gminas for old voivodeship "{old_woj_name}"')
    print(f'  E_ subjects available: {list(agg.cross_tables.keys())}')
    return agg

sampled_old_voiv_record = aggregate_old_voivodeship(db, sampled_old_woj, e_sids)

## Plotting Utilities

In [ ]:
# ── Cell 6: Core plotting function ────────────────────────────────────

# Colour palette — up to ~20 distinguishable colours
PALETTE = (
    plt.cm.tab20.colors[:20]
)

# Census years for reference lines
CENSUS_YEARS = [1988, 2002, 2011, 2021]

def plot_subject_for_record(
    record,
    e_sid: str,
    ax: plt.Axes,
    *,
    dim_idx: int = 0,
    skip_total_label: str | None = 'ogółem',
    is_gmina: bool = True,
):
    """Plot one E_ subject's marginal categories on a single Axes.

    Parameters
    ----------
    record : TERYTRecord or SimpleNamespace
        Must have `.cross_tables` dict.
    e_sid : str
        E_ subject identifier.
    ax : matplotlib Axes
    dim_idx : int
        Which dimension to disaggregate.  0 = first (e.g. age groups).
    skip_total_label : str or None
        Label to skip in the per-category plot (e.g. 'ogółem' = total)
        to avoid cluttering the figure with a line that dwarfs others.
        Set to None to include all.
    is_gmina : bool
        If True, observed points are shown as filled circles (●) and
        estimated points as hollow circles (○).
        If False, all points rendered as hollow diamonds (◇).
        NOTE: Now defaults to True so that observed markers show at
        every territorial level (gmina, powiat, voivodeship).
    """
    years, labels, values, obs_mask = build_category_series(
        record, e_sid, dim_idx=dim_idx
    )
    if len(years) == 0:
        ax.set_title(f'{E_LABELS.get(e_sid, e_sid)}\n(no data)', fontsize=9)
        ax.set_visible(False)
        return

    years_arr = np.array(years)

    # Choose which labels to plot
    plot_idx = []
    for i, lbl in enumerate(labels):
        if skip_total_label and lbl.lower() == skip_total_label.lower():
            continue
        plot_idx.append(i)

    # If skipping the total removed everything, plot all
    if not plot_idx:
        plot_idx = list(range(len(labels)))

    for k, idx in enumerate(plot_idx):
        colour = PALETTE[k % len(PALETTE)]
        lbl = labels[idx]
        ys = values[idx, :]

        # Draw the connecting line
        ax.plot(years_arr, ys, color=colour, linewidth=0.9, alpha=0.8)

        if is_gmina:
            # Observed points: filled circle
            if obs_mask.any():
                ax.scatter(
                    years_arr[obs_mask], ys[obs_mask],
                    marker='o', s=22, color=colour, zorder=3,
                    label=f'{lbl} (obs)' if k == 0 or len(plot_idx) <= 6 else None,
                )
            # Estimated points: hollow circle
            est_mask = ~obs_mask
            if est_mask.any():
                ax.scatter(
                    years_arr[est_mask], ys[est_mask],
                    marker='o', s=22, facecolors='none',
                    edgecolors=colour, linewidths=0.8, zorder=3,
                    label=f'{lbl} (est)' if k == 0 or len(plot_idx) <= 6 else None,
                )
        else:
            # All aggregated → hollow diamonds
            ax.scatter(
                years_arr, ys,
                marker='D', s=16, facecolors='none',
                edgecolors=colour, linewidths=0.7, zorder=3,
            )

    # Light vertical lines at census years
    section = get_section(e_sid)
    section_yrs = SECTION_YEARS[section]
    for cy in CENSUS_YEARS:
        if cy in section_yrs:
            ax.axvline(cy, color='grey', linestyle=':', linewidth=0.5, alpha=0.5)

    ax.set_title(E_LABELS.get(e_sid, e_sid), fontsize=9)
    ax.set_xlabel('Year', fontsize=8)
    ax.set_ylabel('Count', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=8))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f'{x/1000:.0f}k' if abs(x) >= 1000 else f'{x:.0f}'
    ))

    # Compact legend only when few categories
    if len(plot_idx) <= 6:
        # Build label handles manually for clean legend
        handles = []
        for k, idx in enumerate(plot_idx):
            colour = PALETTE[k % len(PALETTE)]
            handles.append(plt.Line2D(
                [0], [0], color=colour, linewidth=1.2,
                marker='o', markersize=4, label=labels[idx],
            ))
        ax.legend(handles=handles, fontsize=5, loc='best',
                  framealpha=0.7, ncol=1)


def plot_total_for_record(
    record,
    e_sid: str,
    ax: plt.Axes,
    *,
    is_gmina: bool = True,
):
    """Plot total population from the E_ cross table (summed across all cells).

    NOTE: is_gmina now defaults to True so observed markers show at
    every territorial level.
    """
    years, totals, obs_mask = build_time_series(record, e_sid)
    if len(years) == 0:
        ax.set_visible(False)
        return

    years_arr = np.array(years)
    ax.plot(years_arr, totals, color='black', linewidth=1.2)

    if is_gmina:
        if obs_mask.any():
            ax.scatter(years_arr[obs_mask], totals[obs_mask],
                       marker='o', s=30, color='steelblue',
                       label='Observed', zorder=3)
        est = ~obs_mask
        if est.any():
            ax.scatter(years_arr[est], totals[est],
                       marker='o', s=30, facecolors='none',
                       edgecolors='steelblue', linewidths=1,
                       label='Estimated', zorder=3)
        ax.legend(fontsize=7, loc='best')
    else:
        ax.scatter(years_arr, totals,
                   marker='D', s=20, facecolors='none',
                   edgecolors='black', linewidths=0.8, zorder=3)

    # Light vertical lines at census years
    section = get_section(e_sid)
    section_yrs = SECTION_YEARS[section]
    for cy in CENSUS_YEARS:
        if cy in section_yrs:
            ax.axvline(cy, color='grey', linestyle=':', linewidth=0.5, alpha=0.5)

    ax.set_title(f'{E_LABELS.get(e_sid, e_sid)} — Total', fontsize=9)
    ax.set_xlabel('Year', fontsize=8)
    ax.set_ylabel('Total count', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=8))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f'{x/1000:.0f}k' if abs(x) >= 1000 else f'{x:.0f}'
    ))

print('Plotting utilities defined.')

In [ ]:
# ── Cell 7: Master plotting function for one territory ─────────────────

def plot_all_subjects(
    record,
    territory_label: str,
    is_gmina: bool = False,
    if_save_fig: bool = False,
):
    """Create a figure with sub-plots for every E_ subject on `record`.

    Layout: two columns —
      left : marginal categories (category lines)
      right: total population (single line)

    Only subjects that actually exist on the record are plotted.
    """
    present_sids = [sid for sid in e_sids if sid in record.cross_tables]
    if not present_sids:
        print(f'  ⚠ No E_ subjects found on {territory_label}')
        return

    n = len(present_sids)
    fig, axes = plt.subplots(n, 2, figsize=(14, 3.2 * n), squeeze=False)
    fig.suptitle(
        f'{territory_label}',
        fontsize=12, fontweight='bold', y=0.96,
    )

    for i, sid in enumerate(present_sids):
        plot_subject_for_record(
            record, sid, axes[i, 0],
            dim_idx=0, is_gmina=is_gmina,
        )
        plot_total_for_record(
            record, sid, axes[i, 1],
            is_gmina=is_gmina,
        )

    fig.tight_layout(rect=[0, 0, 1, 0.97])
    if if_save_fig:
        fig.savefig(plot_root / f'{record.teryt_id}_E_subjects.png', dpi=300)
    plt.show()

print('Master plot function defined.')

## 1.  Random Gmina

In [ ]:
# ── Cell 8: Plot — sampled gmina ──────────────────────────────────────
plot_all_subjects(
    sampled_gmina,
    f'Gmina: {sampled_gmina.name}  ({sampled_gmina.teryt_id})',
    is_gmina=True,
)

for rec in db._records.values():
    is_gmina_powiat = False
    if rec.level == LEVEL_POWIAT:
        child = db.get_by_teryt_id(rec.teryt_id).children_ids
        if all(len(lst) == 1 for lst in child.values()):
            try:
                child = db.get_by_teryt_id(rec.teryt_id).children_ids[2024][0]
                record = db.get_by_teryt_id(child)
                plot_all_subjects(
                    record,
                    f'M. na prawach powiatu: {record.name}  ({record.teryt_id})',
                    is_gmina=True,
                    if_save_fig=True,
                )
            except KeyError:
                continue
          

## 2.  Random Powiat

In [ ]:
# ── Cell 9: Plot — sampled powiat ─────────────────────────────────────
plot_all_subjects(
    sampled_powiat,
    f'Powiat: {sampled_powiat.name}  ({sampled_powiat.teryt_id})',
    is_gmina=True,
)

In [ ]:
sampled_gmina = db.get_by_teryt_id('1431000')
plot_all_subjects(
        sampled_gmina,
        f'Gmina: {sampled_gmina.name}  ({sampled_gmina.teryt_id})',
        is_gmina=True,  # show observed markers at voivodeship level too
        )

## 3.  Random New Voivodeship (post-1999)

In [ ]:
# ── Cell 10: Plot — sampled new voivodeship ───────────────────────────

for voiv in db._by_voivodeship.keys():
    sampled_voiv = db.get_by_teryt_id(voiv + "00000")
    plot_all_subjects(
        sampled_voiv,
        f'New Voivodeship: {sampled_voiv.name}  ({sampled_voiv.teryt_id})',
        is_gmina=True,  # show observed markers at voivodeship level too
        if_save_fig=True,
    )

plot_all_subjects(
        sampled_voiv,
        f'New Voivodeship: {sampled_voiv.name}  ({sampled_voiv.teryt_id})',
        is_gmina=True,  # show observed markers at voivodeship level too
        )

## 4.  Random Old Voivodeship (pre-1999)

In [ ]:
# ── Cell 11: Plot — sampled old voivodeship ───────────────────────────
plot_all_subjects(
    sampled_old_voiv_record,
    f'Old Voivodeship: {sampled_old_woj}  (pre-1999, aggregated from gminas)',
    is_gmina=True,  # show observed markers at old-voivodeship level too
)

In [ ]:
# ── Cell 11: Plot — sampled old voivodeship ───────────────────────────
plot_all_subjects(
    db.get_by_teryt_id('0000000'),
    f'Old Voivodeship: {sampled_old_woj}  (pre-1999, aggregated from gminas)',
    is_gmina=True,  # show observed markers at old-voivodeship level too
)

## 5.  Coverage Summary Table

In [ ]:
# ── Cell 12: Summary — subject coverage across all territories ────────
#
# For each of the four sampled territories, show which E_ subjects
# are available, how many years have data, and (for the gmina) which
# years are observed vs estimated.

rows = []
territory_records = [
    ('Gmina', sampled_gmina),
    ('Powiat', sampled_powiat),
    ('New Voivodeship', sampled_voiv),
    ('Old Voivodeship', sampled_old_voiv_record),
]

for terr_label, rec in territory_records:
    for sid in e_sids:
        ct = rec.cross_tables.get(sid)
        if ct is None:
            rows.append({
                'Territory': terr_label,
                'Name': rec.name,
                'Subject': sid,
                'Years w/ data': 0,
                'Observed years': '-',
                'Estimated years': '-',
            })
            continue
        yrs = ct.years_with_data
        n_yrs = len(yrs)
        if terr_label == 'Gmina':
            obs_years = [y for y in yrs if is_year_observed(rec, sid, y)]
            est_years = [y for y in yrs if not is_year_observed(rec, sid, y)]
            rows.append({
                'Territory': terr_label,
                'Name': rec.name,
                'Subject': sid,
                'Years w/ data': n_yrs,
                'Observed years': len(obs_years),
                'Estimated years': len(est_years),
            })
        else:
            rows.append({
                'Territory': terr_label,
                'Name': rec.name,
                'Subject': sid,
                'Years w/ data': n_yrs,
                'Observed years': '(agg)',
                'Estimated years': n_yrs,
            })

summary_df = pd.DataFrame(rows)
display(summary_df)

## 6.  Detailed Gmina Drill-Down

Show the **raw cross-table DataFrames** for the sampled gmina —
one table per E\_ subject for the census years where observed data exists,
plus one estimated year for comparison.

In [ ]:
# ── Cell 13: Drill-down — raw cross tables for sampled gmina ──────────

CENSUS_YEARS = [1988, 2002, 2011, 2021]

for sid in e_sids:
    ct = sampled_gmina.cross_tables.get(sid)
    if ct is None:
        continue
    data_years = ct.years_with_data
    if not data_years:
        continue

    print(f'\n{"=" * 60}')
    print(f'  {sid}  —  dims: {ct.dim_names}  shape: {ct.shape}')
    print(f'  Years with data: {data_years}')
    print(f'{"=" * 60}')

    # Show census years that have data + one estimated year
    show_years = [y for y in CENSUS_YEARS if y in data_years]
    # Pick a non-census year that has data
    estimated_years = [y for y in data_years if y not in CENSUS_YEARS]
    if estimated_years:
        mid = estimated_years[len(estimated_years) // 2]
        show_years.append(mid)

    for y in sorted(show_years):
        obs = is_year_observed(sampled_gmina, sid, y)
        tag = '● OBSERVED' if obs else '○ ESTIMATED'
        print(f'\n  Year {y} — {tag}')
        df_table = ct.get_as_dataframe(y)
        display(df_table)

In [ ]:
# ── DIAG: Compact proportion distortion evidence ──
S = ['gim', 'pol', 'wyz', 'zas', 'sro']

tests = [('2406013','Klobuck','2400000'), ('1063011','Skiern','1000000'),
         ('0201011','Boles','0200000'), ('0662011','Kraków','0600000')]

for tid, name, vtid in tests:
    r = db._records.get(tid)
    vr = db._records.get(vtid)
    if not r: continue
    ect = r.cross_tables.get('E_educ_2000')
    vct = vr.cross_tables.get('M_educ_2000') if vr else None
    if not ect: continue
    print(f"\n{name} ({tid}) in {vtid}:")
    for yr in [2001, 2002, 2003]:
        tbl = ect.tables.get(yr)
        if tbl is None: continue
        obs = "●" if is_year_observed(r, 'E_educ_2000', yr) else "○"
        sh = tbl/np.nansum(tbl)*100
        print(f"  G {yr}{obs} " + " ".join(f"{S[i]}={sh[i]:5.1f}" for i in range(5)))
    if vct:
        for yr in [2001, 2002]:
            vtbl = vct.tables.get(yr)
            if vtbl is not None:
                vsh = vtbl/np.nansum(vtbl)*100
                print(f"  V {yr}  " + " ".join(f"{S[i]}={vsh[i]:5.1f}" for i in range(5)))

In [ ]:
# ── DEEP DIAGNOSTIC: Analyze all validation FAILs ──
import importlib, sys
sys.path.insert(0, TOOLS_PATH)
import demographic_estimator as de
importlib.reload(de)
from demographic_estimator import DemographicEstimator, ANCHOR_SUBJECTS, E_SUBJECT_NAMES, PREDICTION_2000_RANGE, PREDICTION_1990_RANGE, RODZ_AGGREGATION_SET, _get_aggregation_children
import numpy as np
from collections import defaultdict

est = DemographicEstimator(db, verbose=False)

ALL_E = [
    'E_age_sex_2000', 'E_age_sex_1990',
    'E_educ_2000', 'E_educ_1990',
    'E_educ_sex_2000', 'E_educ_sex_1990',
    'E_hh_size_2000', 'E_hh_size_1990',
]

# Run validation for each, collect all FAIL rows
all_fails = []
for e_sid in ALL_E:
    df = est.validate_results(e_sid)
    fails = df[df['status'] == 'FAIL'].copy()
    fails['e_subject'] = e_sid
    all_fails.append(fails)
    print(f"{e_sid}: {len(fails)} FAILs")

all_fails_df = pd.concat(all_fails, ignore_index=True)
print(f"\nTotal FAILs: {len(all_fails_df)}")
print(f"\nFAILs by (e_subject, check):")
for (esid, chk), grp in all_fails_df.groupby(['e_subject', 'check']):
    print(f"  {esid} / {chk}: {len(grp)}")
    # Show unique teryt_ids
    unique_tids = sorted(grp['teryt_id'].unique())
    unique_years = sorted(grp['year'].unique())
    print(f"    teryt_ids: {unique_tids}")
    print(f"    years: {unique_years}")
    # Show first few details
    for _, row in grp.head(5).iterrows():
        print(f"    {row['teryt_id']} {row['name']} {row['year']}: {row['detail']}")


In [ ]:
# ── ROOT CAUSE 1: E_age_sex_2000 marginal_consistency (36 FAILs) ──
# All in PODLASKIE (2000000), 2005000 (sokólski), and 2005022 (Dąbrowa Białostocka)
# Let's inspect the actual M_age_sex source data for these records

print("="*70)
print("ROOT CAUSE 1: E_age_sex_2000 / marginal_consistency")
print("="*70)
for tid in ['2005022', '2005000', '2000000']:
    rec = db._records.get(tid)
    if not rec: continue
    print(f"\n--- {tid} {rec.name} (level={rec.level}) ---")
    # Check M_age_sex source
    m_ct = rec.cross_tables.get('M_age_sex')
    e_ct = rec.cross_tables.get('E_age_sex_2000')
    if m_ct:
        print(f"  M_age_sex shape={m_ct.shape}, dim_names={m_ct.dim_names}")
        for yr in [2015, 2016, 2017, 2018, 2023, 2024]:
            tbl = m_ct.tables.get(yr)
            if tbl is not None:
                row_sums = tbl[:-1, :].sum(axis=0)  # sum non-ogółem rows
                og_row = tbl[-1, :] if tbl.shape[0] > 0 else None
                # Check if any dim has ogółem
                # Identify ogółem
                labels_n1 = m_ct.dim_labels.get(m_ct.dim_names[0], [])
                labels_n2 = m_ct.dim_labels.get(m_ct.dim_names[1], [])
                og1 = [i for i, l in enumerate(labels_n1) if 'ogółem' in l.lower()]
                og2 = [i for i, l in enumerate(labels_n2) if 'ogółem' in l.lower()]
                if og1:
                    og_row_vals = tbl[og1[0], :]
                    non_og_rows = [i for i in range(len(labels_n1)) if i not in og1]
                    sub_sum = tbl[non_og_rows, :].sum(axis=0)
                    diff = og_row_vals - sub_sum
                    max_err = np.max(np.abs(diff))
                    if max_err > 1.0:
                        print(f"    M_ yr={yr}: dim=n1 SOURCE marginal err={max_err:.4f}")
                if og2:
                    og_col_vals = tbl[:, og2[0]]
                    non_og_cols = [i for i in range(len(labels_n2)) if i not in og2]
                    sub_sum_c = tbl[:, non_og_cols].sum(axis=1)
                    diff_c = og_col_vals - sub_sum_c
                    max_err_c = np.max(np.abs(diff_c))
                    if max_err_c > 1.0:
                        print(f"    M_ yr={yr}: dim=n2 SOURCE marginal err={max_err_c:.4f}")

    if e_ct:
        labels_n1 = e_ct.dim_labels.get(e_ct.dim_names[0], [])
        labels_n2 = e_ct.dim_labels.get(e_ct.dim_names[1], [])
        og1 = [i for i, l in enumerate(labels_n1) if 'ogółem' in l.lower()]
        og2 = [i for i, l in enumerate(labels_n2) if 'ogółem' in l.lower()]
        for yr in [2015, 2016, 2017, 2018, 2023, 2024]:
            tbl = e_ct.tables.get(yr)
            if tbl is None: continue
            if og1:
                non_og_rows = [i for i in range(len(labels_n1)) if i not in og1]
                sub_sum = tbl[non_og_rows, :].sum(axis=0)
                diff = tbl[og1[0], :] - sub_sum
                max_err = np.max(np.abs(diff))
                if max_err > 1.0:
                    print(f"    E_ yr={yr}: dim=n1 max_err={max_err:.4f}")
            if og2:
                non_og_cols = [i for i in range(len(labels_n2)) if i not in og2]
                sub_sum_c = tbl[:, non_og_cols].sum(axis=1)
                diff_c = tbl[:, og2[0]] - sub_sum_c
                max_err_c = np.max(np.abs(diff_c))
                if max_err_c > 1.0:
                    print(f"    E_ yr={yr}: dim=n2 max_err={max_err_c:.4f}")

    # Does this unit have code_by_year changes?
    if hasattr(rec, 'code_by_year') and rec.code_by_year:
        print(f"  code_by_year: {dict(rec.code_by_year)}")


In [ ]:
# ── ROOT CAUSE 2: E_age_sex_1990 hierarchical_consistency (12 FAILs) ──
# Powiats: 1008000, 1207000, 1211000, 2413000, 2415000 in years 1995-1997
print("="*70)
print("ROOT CAUSE 2: E_age_sex_1990 / hierarchical_consistency")
print("="*70)

for ptid in ['1008000', '1207000', '1211000', '2413000', '2415000']:
    prec = db._records.get(ptid)
    if not prec: continue
    print(f"\n--- Powiat {ptid} {prec.name} ---")
    pct = prec.cross_tables.get('E_age_sex_1990')
    if not pct: continue
    for yr in [1995, 1996, 1997]:
        ptbl = pct.tables.get(yr)
        if ptbl is None: continue
        # Get children
        children = _get_aggregation_children(prec, db, yr)
        child_sum = np.zeros_like(ptbl)
        n_ch = 0
        for chtid in children:
            chrec = db._records.get(chtid)
            if not chrec: continue
            chct = chrec.cross_tables.get('E_age_sex_1990')
            if not chct: continue
            chtbl = chct.tables.get(yr)
            if chtbl is None: continue
            child_sum += np.nan_to_num(chtbl, nan=0.0)
            n_ch += 1
        diff = child_sum - np.nan_to_num(ptbl, nan=0.0)
        max_diff = np.max(np.abs(diff))
        ptotal = np.nansum(np.abs(ptbl))
        pct_err = 100 * max_diff / ptotal if ptotal > 0 else 0
        print(f"  yr={yr}: n_children={n_ch}, max_diff={max_diff:.1f} ({pct_err:.3f}%)")
        print(f"    powiat_total={np.nansum(ptbl):.0f}, children_total={np.nansum(child_sum):.0f}")
        
        # What are those children? Are there any that changed codes?
        if yr == 1995:
            print(f"    children: {sorted(children)}")
            # Check if all children have data for 1995
            for chtid in sorted(children):
                chrec = db._records.get(chtid)
                if not chrec: continue
                chct = chrec.cross_tables.get('E_age_sex_1990')
                if chct and chct.tables.get(yr) is not None:
                    ch_total = np.nansum(chct.tables[yr])
                else:
                    ch_total = 0
                # Check M_age_sex
                m_ct = chrec.cross_tables.get('M_age_sex')
                m_yrs = sorted(m_ct.tables.keys()) if m_ct else []
                m_has_1995 = yr in m_yrs if m_ct else False
                cby = dict(chrec.code_by_year) if hasattr(chrec, 'code_by_year') and chrec.code_by_year else {}
                code_at_yr = cby.get(yr, chtid)
                flag = " *** CODE DIFFERS" if code_at_yr != chtid else ""
                print(f"      {chtid} {chrec.name}: E_total={ch_total:.0f}, "
                      f"M_has_{yr}={m_has_1995}, code@{yr}={code_at_yr}{flag}")


In [ ]:
# ── RC2 compact: age_sex_1990 hierarchical ──
print("="*70)
print("RC2: E_age_sex_1990 hierarchical — powiat total vs sum(children)")
print("="*70)

for ptid in ['1008000', '1207000', '1211000', '2413000', '2415000']:
    prec = db._records.get(ptid)
    if not prec: continue
    pct = prec.cross_tables.get('E_age_sex_1990')
    if not pct: continue
    print(f"\n--- {ptid} {prec.name} ---")
    for yr in [1994, 1995, 1996, 1997, 1998]:
        ptbl = pct.tables.get(yr)
        if ptbl is None:
            print(f"  yr={yr}: NO DATA")
            continue
        children = _get_aggregation_children(prec, db, yr)
        child_sum = np.zeros_like(ptbl)
        n_ch = n_ch_data = 0
        for chtid in children:
            n_ch += 1
            chrec = db._records.get(chtid)
            if not chrec: continue
            chct = chrec.cross_tables.get('E_age_sex_1990')
            if not chct: continue
            chtbl = chct.tables.get(yr)
            if chtbl is None: continue
            child_sum += np.nan_to_num(chtbl, nan=0.0)
            n_ch_data += 1
        max_diff = np.max(np.abs(child_sum - np.nan_to_num(ptbl, nan=0.0)))
        ptotal = np.nansum(ptbl)
        print(f"  yr={yr}: P_total={ptotal:.0f}, ch_sum={np.nansum(child_sum):.0f}, "
              f"n_ch={n_ch_data}/{n_ch}, max_diff={max_diff:.1f}")

# Now check: are the children the same between 1994 and 1995?
print(f"\n\nChild IDs comparison 1994 vs 1995 vs 1999:")
for ptid in ['1008000']:
    prec = db._records.get(ptid)
    if not prec: continue
    for yr in [1994, 1995, 1999]:
        children = _get_aggregation_children(prec, db, yr)
        print(f"  {ptid} yr={yr}: {sorted(children)}")
    # The powiat table is aggregated from children - who computed it?
    # For 1990 period, E_ powiat data is aggregated from gminas in the estimator
    # Let's check if the powiat itself has M_age_sex data
    m_ct = prec.cross_tables.get('M_age_sex')
    if m_ct:
        m_yrs = sorted(y for y in m_ct.tables if m_ct.tables[y] is not None)
        print(f"  {ptid} M_age_sex years: {m_yrs}")


In [ ]:
# ── RC2 deeper: Why does powiat total differ from sum of children? ──
# The powiat E_ is populated by aggregation in _aggregate_to_powiat
# But children also have E_ data from estimation
# Check: is the powiat E_ data coming from aggregation or hybrid-scaling?

# Look at tarnogórski 2413000 — biggest discrepancy
ptid = '2413000'
prec = db._records[ptid]
print(f"=== {ptid} {prec.name} ===")

# Check E_age_sex_1990 powiat table totals
e_ct = prec.cross_tables.get('E_age_sex_1990')
# Check each child's E_age_sex_1990, and also their E_age_sex_2000 for 1999
children = _get_aggregation_children(prec, db, 1999)
print(f"Children: {children}")

for chtid in sorted(children):
    chrec = db._records.get(chtid)
    if not chrec: continue
    e90 = chrec.cross_tables.get('E_age_sex_1990')
    e00 = chrec.cross_tables.get('E_age_sex_2000')
    m = chrec.cross_tables.get('M_age_sex')
    
    e90_tot_94 = np.nansum(e90.tables[1994]) if e90 and 1994 in e90.tables and e90.tables[1994] is not None else float('nan')
    e90_tot_95 = np.nansum(e90.tables[1995]) if e90 and 1995 in e90.tables and e90.tables[1995] is not None else float('nan')
    e00_tot_99 = np.nansum(e00.tables[1999]) if e00 and 1999 in e00.tables and e00.tables[1999] is not None else float('nan')
    m_tot_95 = np.nansum(m.tables[1995]) if m and 1995 in m.tables and m.tables[1995] is not None else float('nan')
    m_tot_99 = np.nansum(m.tables[1999]) if m and 1999 in m.tables and m.tables[1999] is not None else float('nan')
    pop_95 = chrec.pop.get(pd.Timestamp(1995,1,1), float('nan'))
    pop_99 = chrec.pop.get(pd.Timestamp(1999,1,1), float('nan'))
    
    cby = dict(chrec.code_by_year) if hasattr(chrec, 'code_by_year') and chrec.code_by_year else {}
    code_95 = cby.get(1995, chtid)
    code_flag = " ***" if code_95 != chtid else ""
    
    print(f"  {chtid} {chrec.name}: E90@94={e90_tot_94:.0f}, E90@95={e90_tot_95:.0f}, "
          f"M@95={m_tot_95:.0f}, pop@95={pop_95:.0f}, code@95={code_95}{code_flag}")

# Now check: what does the powiat have at 1995 vs what the sum of children gives
print(f"\nPowiat E_age_sex_1990:")
for yr in [1993, 1994, 1995, 1996, 1997, 1998, 1999]:
    ptbl = e_ct.tables.get(yr)
    if ptbl is not None:
        print(f"  yr={yr}: total={np.nansum(ptbl):.0f}")
    else:
        print(f"  yr={yr}: NO DATA")

# Check how _aggregate_to_powiat works for 1990
print(f"\nPowiat M_age_sex:")
m_p = prec.cross_tables.get('M_age_sex')
if m_p:
    for yr in [1993, 1994, 1995, 1996, 1997, 1998, 1999]:
        mtbl = m_p.tables.get(yr)
        if mtbl is not None:
            print(f"  yr={yr}: M total={np.nansum(mtbl):.0f}")


In [ ]:
# ── RC3: E_educ_2000 hierarchical_consistency — 102 FAILs all at 2011 ──
# Hypothesis: 2011 census powiat-disaggregation step creates gmina tables
# that don't exactly round-trip back to the powiat total.
print("="*70)
print("RC3: E_educ_2000 / hierarchical_consistency — all year=2011")
print("="*70)

rc3_tids = ['0264000', '0663000', '1261000', '1401000', '1402000',
            '1461000', '1462000', '1463000', '1464000', '1465000',
            '2201000', '2261000', '2601000', '2861000', '3261000']

# Pattern analysis: are these ALL powiats? or also voivodeships?
rc3_all = sorted(all_fails_df[(all_fails_df['e_subject']=='E_educ_2000') & 
                               (all_fails_df['check']=='hierarchical_consistency')]['teryt_id'].unique())
pow_cnt = sum(1 for t in rc3_all if t[-1]=='0' and t[2:]!='00000')
voiv_cnt = sum(1 for t in rc3_all if t[2:]=='00000')
city_cnt = sum(1 for t in rc3_all if t[-1] not in ('0',) and t[2:]!='00000')
print(f"Total: {len(rc3_all)} — powiats: {pow_cnt}, voivs: {voiv_cnt}, others: {city_cnt}")

# Check a few representative powiats
for ptid in ['1401000', '1465000', '0264000', '2201000']:
    prec = db._records.get(ptid)
    if not prec: continue
    pct = prec.cross_tables.get('E_educ_2000')
    if not pct: continue
    ptbl = pct.tables.get(2011)
    if ptbl is None: continue
    
    children = _get_aggregation_children(prec, db, 2011)
    child_sum = np.zeros_like(ptbl)
    n_ch = 0
    ch_details = []
    for chtid in children:
        chrec = db._records.get(chtid)
        if not chrec: continue
        chct = chrec.cross_tables.get('E_educ_2000')
        if not chct: continue
        chtbl = chct.tables.get(2011)
        if chtbl is None: continue
        child_sum += np.nan_to_num(chtbl, nan=0.0)
        n_ch += 1
    
    diff = child_sum - np.nan_to_num(ptbl, nan=0.0)
    max_diff = np.max(np.abs(diff))
    ptotal = np.nansum(np.abs(ptbl))
    pct_err = 100*max_diff/ptotal if ptotal>0 else 0
    
    # Also check what's the M_ source at 2011
    m_ct = prec.cross_tables.get('M_educ_2000')
    m_tbl = m_ct.tables.get(2011) if m_ct else None
    m_total = np.nansum(m_tbl) if m_tbl is not None else float('nan')
    
    print(f"\n  {ptid} {prec.name}:")
    print(f"    E_  ptotal={np.nansum(ptbl):.0f}, ch_sum={np.nansum(child_sum):.0f}, "
          f"max_diff={max_diff:.1f} ({pct_err:.3f}%)")
    print(f"    M_  at 2011: {m_total:.0f}")
    # Per-cell comparison
    print(f"    E_powiat: {np.nan_to_num(ptbl, nan=0.0)}")
    print(f"    ch_sum  : {child_sum}")
    print(f"    diff    : {diff}")
    
    # Check how E_educ_2000 powiat was created: was it pure aggregation or hybrid-scaled?
    print(f"    Children (n={n_ch}): {sorted(children)[:5]}...")

# Special check: was 2011 data at powiat level from P3309 (powiat-level census)?
# The 2011 census for education was at POWIAT level (P3309), disaggregated to gmina
print(f"\n\n--- M_educ_2000 powiat data for 1401000 around 2011: ---")
prec = db._records['1401000']
m_ct = prec.cross_tables.get('M_educ_2000')
if m_ct:
    for yr in [2009, 2010, 2011, 2012, 2013]:
        mtbl = m_ct.tables.get(yr)
        if mtbl is not None:
            print(f"  yr={yr}: {np.nan_to_num(mtbl, nan=0.0)}")
        else:
            print(f"  yr={yr}: None")


In [ ]:
# ── RC4/5: educ_sex marginal_consistency: dim=n2 failures at 2002 ──
# 12 identical teryt_ids for both E_educ_sex_2000 and E_educ_sex_1990
# All at year 2002 (census year)
print("="*70)
print("RC4/5: E_educ_sex marginal_consistency (n2) at year 2002")
print("="*70)

fail_tids = ['0407022', '0408022', '1413022', '1415022', '1416092']
for tid in fail_tids:
    rec = db._records.get(tid)
    if not rec: continue
    print(f"\n--- {tid} {rec.name} (level={rec.level}, rodz={rec.rodz}) ---")
    
    # Check M_educ_sex_2000 source at 2002
    m_ct = rec.cross_tables.get('M_educ_sex_2000')
    if m_ct:
        mtbl = m_ct.tables.get(2002)
        if mtbl is not None:
            print(f"  M_educ_sex_2000 @2002 shape={m_ct.shape}:")
            print(f"    dim_names={m_ct.dim_names}")
            labels_n1 = m_ct.dim_labels.get(m_ct.dim_names[0], [])
            labels_n2 = m_ct.dim_labels.get(m_ct.dim_names[1], [])
            print(f"    dim_labels[n1]={labels_n1}")
            print(f"    dim_labels[n2]={labels_n2}")
            # Check n2 marginal
            og2_idx = [i for i, l in enumerate(labels_n2) if 'ogółem' in l.lower()]
            if og2_idx:
                non_og2 = [i for i in range(len(labels_n2)) if i not in og2_idx]
                og_col = mtbl[:, og2_idx[0]]
                sub_sum = mtbl[:, non_og2].sum(axis=1)
                diff = og_col - sub_sum
                max_err = np.max(np.abs(diff))
                print(f"    M_ source n2 marginal err: {max_err:.4f}")
                if max_err > 1.0:
                    # Show per-row breakdown
                    for r in range(len(labels_n1)):
                        if abs(diff[r]) > 0.5:
                            print(f"      row '{labels_n1[r]}': og={og_col[r]:.0f}, sum={sub_sum[r]:.0f}, diff={diff[r]:.0f}")
    
    # Check E_educ_sex_2000 at 2002
    e_ct = rec.cross_tables.get('E_educ_sex_2000')
    if e_ct:
        etbl = e_ct.tables.get(2002)
        if etbl is not None:
            labels_n2e = e_ct.dim_labels.get(e_ct.dim_names[1], [])
            og2_idx = [i for i, l in enumerate(labels_n2e) if 'ogółem' in l.lower()]
            if og2_idx:
                non_og2 = [i for i in range(len(labels_n2e)) if i not in og2_idx]
                og_col = etbl[:, og2_idx[0]]
                sub_sum = etbl[:, non_og2].sum(axis=1)
                diff = og_col - sub_sum
                max_err = np.max(np.abs(diff))
                print(f"    E_ n2 marginal err @2002: {max_err:.4f}")
    
    # Check M_educ_sex_1990 at 2002
    m90_ct = rec.cross_tables.get('M_educ_sex_1990')
    if m90_ct:
        m90tbl = m90_ct.tables.get(2002)
        if m90tbl is not None:
            labels_n2m = m90_ct.dim_labels.get(m90_ct.dim_names[1], [])
            og2_idx = [i for i, l in enumerate(labels_n2m) if 'ogółem' in l.lower()]
            if og2_idx:
                non_og2 = [i for i in range(len(labels_n2m)) if i not in og2_idx]
                og_col = m90tbl[:, og2_idx[0]]
                sub_sum = m90tbl[:, non_og2].sum(axis=1)
                diff = og_col - sub_sum
                max_err = np.max(np.abs(diff))
                print(f"    M_educ_sex_1990 n2 marginal err @2002: {max_err:.4f}")
                if max_err > 1.0:
                    for r in range(m90tbl.shape[0]):
                        if abs(diff[r]) > 0.5:
                            labels_n1m = m90_ct.dim_labels.get(m90_ct.dim_names[0], [])
                            print(f"      row '{labels_n1m[r]}': og={og_col[r]:.0f}, sum={sub_sum[r]:.0f}, diff={diff[r]:.0f}")


---

### Notes

- Re-run **Cell 4** (random sampling) to inspect a different set of
  territories.  Remove the `random.seed(SEED)` line for truly random
  draws each time.
- The *old voivodeship* plots may show small discrepancies from the
  new voivodeship totals, because the pre-1999 administrative borders
  do not align with post-1999 voivodeship boundaries.
- Observed/estimated markers on gmina plots derive from whether the
  **source M\_ anchor** has real census data at that (territory, year)
  pair.  They are not persisted in the database pickle and are
  reconstructed on the fly.